# Dataset Builder (HDF5 output)

Source data:
- **EEGDenoiseNet** (Zhang et al., 2021) — 4514 EEG, 3400 EOG, 5598 EMG @ 256 Hz
- **MIT-BIH Arrhythmia Database** — ECG @ 360 Hz → 45 Hz LPF → 256 Hz → 2-second segments

Artifact combinations (7 total, paper Table I / §IV-A):

| File prefix | Artifacts |
|---|---|
| `emg` | EMG only |
| `eog` | EOG only |
| `ecg` | ECG only |
| `emg_eog` | EMG + EOG |
| `emg_ecg` | EMG + ECG |
| `eog_ecg` | EOG + ECG |
| `emg_eog_ecg` | EMG + EOG + ECG |

## Output format — HDF5 (`.h5`)

**21 files total: 7 combos × 3 splits.** Each file holds both arrays together:

```
{combo}_{split}.h5
  /clean   float32  (N, 512)   ← normalised ground-truth EEG
  /noisy   float32  (N, 512)   ← normalised contaminated EEG
```

Row `i` of `/clean` is always the ground-truth for row `i` of `/noisy`.

**Why HDF5 instead of CSV?**  
Storing 512 clean + 512 noisy samples per row would produce a 1025-column CSV that is
slow to read and write. HDF5 keeps both arrays in one file, applies gzip compression
(3–5× size reduction), and loads in a single call:

```python
import h5py
with h5py.File('emg_train.h5', 'r') as f:
    clean = f['clean'][:]   # (N, 512) float32 — ground-truth EEG
    noisy = f['noisy'][:]   # (N, 512) float32 — contaminated EEG
```

Split: **80% train / 10% val / 10% test** (paper §IV-A, Table I).  
SNR range: **−7 dB to +2 dB** (10 levels, step 1 dB).  
Normalisation: `x̂ = x/σ_y`, `ŷ = y/σ_y` where `σ_y = std(contaminated)`.

---
**Google Colab:** Runtime → Run all  
**Kaggle:** Session → Run all  
All source data is downloaded automatically.

## 0 — Install dependencies

In [21]:
!git clone https://github.com/ncclabsustech/EEGdenoiseNet.git

fatal: destination path 'EEGdenoiseNet' already exists and is not an empty directory.


In [ ]:
# !unzip mit-bih.zip

In [ ]:
!pip install wfdb scipy

In [ ]:
import subprocess, sys

def pip(*pkgs):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *pkgs])

pip('wfdb', 'scipy', 'numpy', 'pandas', 'h5py', 'tqdm')
print('Dependencies ready')

Dependencies ready


In [ ]:
import gc, warnings, zipfile, urllib.request
from math import gcd
from pathlib import Path

import h5py
import numpy as np
import pandas as pd
import scipy.signal as ss
import wfdb
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')
print('Imports OK')

Imports OK


## 1 — Configuration

In [ ]:
# ================================================================
#  USER CONFIGURATION  — edit these values if needed
# ================================================================

EEGDENOISENET_DIR = Path('/content/EEGdenoiseNet/data')   # downloaded in Cell 2
OUT_DIR           = Path('./dataset1_h5')          # output folder

# Signal parameters
TARGET_FS   = 256          # Hz — common rate for all signals
SEG_SEC     = 2            # seconds per segment
SEGMENT_LEN = TARGET_FS * SEG_SEC   # 512 samples

# MIT-BIH ECG preprocessing (paper §IV-A)
MITBIH_FS        = 360     # original MIT-BIH sampling rate
MITBIH_LPF_HZ    = 45.0   # low-pass cutoff (Hz) before resampling
MITBIH_LPF_ORDER = 5      # Butterworth filter order

# Dataset construction
SNR_LEVELS  = list(range(-7, 3))   # [-7,-6,...,2] dB  (10 levels)
TRAIN_RATIO = 0.80
VAL_RATIO   = 0.10
# TEST_RATIO  = 0.10  (implicit remainder)
RANDOM_SEED = 42

# All 48 MIT-BIH record IDs
MITBIH_RECORDS = [
    '100','101','102','103','104','105','106','107','108','109',
    '111','112','113','114','115','116','117','118','119','121',
    '122','123','124','200','201','202','203','205','207','208',
    '209','210','212','213','214','215','217','219','220','221',
    '222','223','228','230','231','232','233','234'
]

# 7 artifact combinations (paper §IV-A)
ARTIFACT_COMBOS = [
    ('emg',),
    ('eog',),
    ('ecg',),
    ('emg', 'eog'),
    ('emg', 'ecg'),
    ('eog', 'ecg'),
    ('emg', 'eog', 'ecg'),
]

OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f'Segment : {SEGMENT_LEN} samples  ({SEG_SEC} s x {TARGET_FS} Hz)')
print(f'SNR     : {SNR_LEVELS} dB')
print(f'Output  : {OUT_DIR.resolve()}')
print(f'Format  : HDF5 -- /clean (N,512) float32  and  /noisy (N,512) float32')

Segment : 512 samples  (2 s x 256 Hz)
SNR     : [-7, -6, -5, -4, -3, -2, -1, 0, 1, 2] dB
Output  : /content/dataset1_h5
Format  : HDF5 -- /clean (N,512) float32  and  /noisy (N,512) float32


## 2 — Download & load EEGDenoiseNet

In [ ]:
EEGDENOISENET_DIR.mkdir(parents=True, exist_ok=True)

EEGDN_URLS = {
    'EEG_all_epochs.npy': 'https://github.com/ncclabsustech/EEGdenoiseNet/raw/main/data/EEG_all_epochs.npy',
    'EOG_all_epochs.npy': 'https://github.com/ncclabsustech/EEGdenoiseNet/raw/main/data/EOG_all_epochs.npy',
    'EMG_all_epochs.npy': 'https://github.com/ncclabsustech/EEGdenoiseNet/raw/main/data/EMG_all_epochs.npy',
}

for fname, url in EEGDN_URLS.items():
    dest = EEGDENOISENET_DIR / fname
    if dest.exists():
        print(f'  Cached  : {fname}')
    else:
        print(f'  Fetching: {fname} ...', end=' ', flush=True)
        urllib.request.urlretrieve(url, dest)
        print(f'({dest.stat().st_size / 1e6:.1f} MB)')


def load_npy_256hz(path: Path, name: str) -> np.ndarray:
    """
    Load a .npy segment array and guarantee 256 Hz (512 samples/seg).
    EMG is distributed at 512 Hz (1024 samples) and is resampled down.
    """
    arr = np.load(path).astype(np.float64)
    if arr.ndim == 1:
        arr = arr.reshape(1, -1)
    n_segs, n_samples = arr.shape

    if n_samples == SEGMENT_LEN:
        print(f'  {name:<6}: {n_segs:>5} segs x {n_samples} samples  (256 Hz, no resample)')
        return arr

    if n_samples == 1024:   # 512 Hz version
        print(f'  {name:<6}: {n_segs:>5} segs x {n_samples} samples  (512 Hz -> resampling to 256 Hz)', end=' ')
        arr = ss.resample(arr, SEGMENT_LEN, axis=1)
        print('done')
        return arr

    raise ValueError(f'{name}: unexpected sample count per segment: {n_samples}')


print('\nLoading EEGDenoiseNet arrays ...')
eeg = load_npy_256hz(EEGDENOISENET_DIR / 'EEG_all_epochs.npy', 'EEG')
eog = load_npy_256hz(EEGDENOISENET_DIR / 'EOG_all_epochs.npy', 'EOG')
emg = load_npy_256hz(EEGDENOISENET_DIR / 'EMG_all_epochs.npy', 'EMG')

assert eeg.shape[1] == eog.shape[1] == emg.shape[1] == SEGMENT_LEN
print('EEGDenoiseNet arrays ready')

  Cached  : EEG_all_epochs.npy
  Cached  : EOG_all_epochs.npy
  Cached  : EMG_all_epochs.npy

Loading EEGDenoiseNet arrays ...
  EEG   :  4514 segs x 512 samples  (256 Hz, no resample)
  EOG   :  3400 segs x 512 samples  (256 Hz, no resample)
  EMG   :  5598 segs x 512 samples  (256 Hz, no resample)
EEGDenoiseNet arrays ready


## 3 — Download & preprocess MIT-BIH ECG

In [ ]:
# Butterworth LPF coefficients (applied at 360 Hz BEFORE resampling)
b_lpf, a_lpf = ss.butter(
    MITBIH_LPF_ORDER,
    MITBIH_LPF_HZ / (MITBIH_FS / 2.0),
    btype='low'
)

# Integer resampling ratio 360->256 Hz  (gcd=8, up=32, down=45)
_g   = gcd(TARGET_FS, MITBIH_FS)
UP   = TARGET_FS // _g     # 32
DOWN = MITBIH_FS // _g     # 45

print(f'ECG preprocessing:')
print(f'  LPF      : Butterworth order={MITBIH_LPF_ORDER}, cutoff={MITBIH_LPF_HZ} Hz at {MITBIH_FS} Hz')
print(f'  Resample : {MITBIH_FS} Hz -> {TARGET_FS} Hz  (polyphase up={UP}, down={DOWN})')
print(f'  Segment  : {SEG_SEC} s non-overlapping  ({SEGMENT_LEN} samples @ {TARGET_FS} Hz)')
print(f'\nProcessing {len(MITBIH_RECORDS)} MIT-BIH records (auto-download from PhysioNet) ...')

ecg_segments, failed = [], []

for rec_id in tqdm(MITBIH_RECORDS, desc='MIT-BIH'):
    try:
        rec = wfdb.rdrecord(rec_id, pn_dir='mitdb')
        sig = rec.p_signal[:, 0].astype(np.float64)   # MLII lead
        sig = np.where(np.isfinite(sig), sig, 0.0)

        sig = ss.filtfilt(b_lpf, a_lpf, sig)          # 45 Hz LPF @ 360 Hz
        sig = ss.resample_poly(sig, UP, DOWN)          # 360 -> 256 Hz

        n_segs = len(sig) // SEGMENT_LEN
        for k in range(n_segs):
            seg = sig[k * SEGMENT_LEN : (k + 1) * SEGMENT_LEN]
            if np.all(np.isfinite(seg)):
                ecg_segments.append(seg)

    except Exception as exc:
        failed.append((rec_id, str(exc)))

if failed:
    print(f'WARNING: {len(failed)} record(s) skipped:')
    for r, e in failed:
        print(f'  {r}: {e}')

ecg = np.array(ecg_segments, dtype=np.float64)
print(f'\nECG : {ecg.shape[0]:,} segments x {ecg.shape[1]} samples  ({TARGET_FS} Hz)')

ECG preprocessing:
  LPF      : Butterworth order=5, cutoff=45.0 Hz at 360 Hz
  Resample : 360 Hz -> 256 Hz  (polyphase up=32, down=45)
  Segment  : 2 s non-overlapping  (512 samples @ 256 Hz)

Processing 48 MIT-BIH records (auto-download from PhysioNet) ...


MIT-BIH:   0%|          | 0/48 [00:00<?, ?it/s]


ECG : 43,296 segments x 512 samples  (256 Hz)


## 4 — Signal helpers

In [ ]:
def rms(x: np.ndarray) -> float:
    """Root-mean-square  (EEGDenoiseNet Eq. 3)."""
    return float(np.sqrt(np.mean(x ** 2)))


def compute_alpha(eeg_seg: np.ndarray, art_seg: np.ndarray, snr_db: float) -> float:
    """
    Mixing coefficient alpha (CT-DCENet Eq. 7 / EEGDenoiseNet Eq. 2):

        SNR = 10*log10( RMS(xP) / RMS(alpha*n) )
        =>   alpha = RMS(xP) / ( RMS(n) * 10^(SNR/10) )

    Note: the papers use 10*log10 of an RMS (amplitude) ratio,
    so the exponent is SNR/10, not SNR/20.
    """
    r_n = rms(art_seg)
    if r_n == 0.0:
        return 0.0
    return rms(eeg_seg) / (r_n * 10 ** (snr_db / 10.0))


def mix_signals(eeg_seg: np.ndarray,
                artifact_segs: list,
                snr_db: float) -> np.ndarray:
    """
    Contaminated EEG = xP + sum(alpha_k * n_k)  (CT-DCENet Eq. 7).
    Each artifact component is independently scaled to the same SNR target.
    """
    contamination = np.zeros_like(eeg_seg)
    for art in artifact_segs:
        contamination += compute_alpha(eeg_seg, art, snr_db) * art
    return eeg_seg + contamination


def normalise_pair(clean: np.ndarray, noisy: np.ndarray):
    """
    EEGDenoiseNet Eq. 4:  x_hat = x / sigma_y ,  y_hat = y / sigma_y
    sigma_y is the std of the CONTAMINATED segment.
    Returns (clean_norm, noisy_norm) as float32.
    """
    sigma = float(np.std(noisy))
    if sigma == 0.0:
        return clean.astype(np.float32), noisy.astype(np.float32)
    return (clean / sigma).astype(np.float32), (noisy / sigma).astype(np.float32)


def split_indices(n: int, rng: np.random.Generator):
    """Reproducible 80 / 10 / 10 shuffle-split."""
    idx     = rng.permutation(n)
    n_train = int(np.floor(TRAIN_RATIO * n))
    n_val   = int(np.floor(VAL_RATIO   * n))
    return idx[:n_train], idx[n_train : n_train + n_val], idx[n_train + n_val:]


print('Signal helpers defined')

Signal helpers defined


## 5 — Core builder: HDF5 files with `/clean` and `/noisy` datasets

For each artifact combination and each split, **one HDF5 file** is written:

```
{combo}_{split}.h5
  /clean   float32  (N_rows, 512)   normalised ground-truth EEG
  /noisy   float32  (N_rows, 512)   normalised contaminated EEG
```

Both datasets live in the same file, so alignment is guaranteed by construction.
File-level and dataset-level HDF5 attributes store all metadata.

In [ ]:
def build_and_save(
        eeg_arr:    np.ndarray,
        art_names:  tuple,
        art_arrays: dict,
        snr_levels: list,
        out_dir:    Path,
        rng:        np.random.Generator,
) -> dict:
    """
    Generate train / val / test HDF5 files for one artifact combination.

    Each file holds /clean (N,512) and /noisy (N,512) as float32 arrays.
    Row i of /clean is the ground-truth for row i of /noisy.

    Procedure (paper §IV-A):
      1. Split EEG and each artifact pool independently 80/10/10.
      2. Within each split, randomly pair each EEG segment with one segment
         from each artifact pool.
      3. For every pair, generate one contaminated segment per SNR level.
      4. Normalise (clean, noisy) by std(noisy).
      5. Write both arrays to a single compressed HDF5 file.

    Returns {split: n_rows} for the summary table.
    """
    combo = '_'.join(art_names)
    n_eeg = eeg_arr.shape[0]
    n_snr = len(snr_levels)

    # --- independent splits for EEG and every artifact pool ---
    eeg_tr, eeg_va, eeg_te = split_indices(n_eeg, rng)

    art_splits = {}
    for name in art_names:
        art_splits[name] = split_indices(art_arrays[name].shape[0], rng)

    splits_map = {
        'train': (eeg_tr, {n: art_splits[n][0] for n in art_names}),
        'val':   (eeg_va, {n: art_splits[n][1] for n in art_names}),
        'test':  (eeg_te, {n: art_splits[n][2] for n in art_names}),
    }

    row_counts = {}

    for split_name, (e_idx, a_idx_map) in splits_map.items():
        eeg_split = eeg_arr[e_idx]    # (n_e, 512)
        n_e       = len(e_idx)

        # random partner from each artifact pool for every EEG segment
        paired = {}
        for name in art_names:
            pool     = art_arrays[name][a_idx_map[name]]
            pick_idx = rng.integers(0, len(pool), size=n_e)
            paired[name] = pool[pick_idx]   # (n_e, 512)

        total     = n_e * n_snr
        clean_buf = np.empty((total, SEGMENT_LEN), dtype=np.float32)
        noisy_buf = np.empty((total, SEGMENT_LEN), dtype=np.float32)

        row = 0
        for i in range(n_e):
            x         = eeg_split[i]
            arts_list = [paired[name][i] for name in art_names]
            for snr in snr_levels:
                y              = mix_signals(x, arts_list, snr_db=snr)
                x_n, y_n       = normalise_pair(x, y)
                clean_buf[row] = x_n
                noisy_buf[row] = y_n
                row           += 1

        assert row == total, f'Row mismatch: {row} != {total}'

        # --- write one HDF5 file with /clean and /noisy inside ---
        h5_path = out_dir / f'{combo}_{split_name}.h5'
        chunk   = (min(256, total), SEGMENT_LEN)

        with h5py.File(h5_path, 'w') as hf:

            # /clean
            ds_c = hf.create_dataset(
                'clean', data=clean_buf, dtype='float32',
                compression='gzip', compression_opts=4, chunks=chunk,
            )
            ds_c.attrs['description']     = (
                'Normalised ground-truth EEG. '
                'shape=(N_segments, 512). '
                'Row i pairs with /noisy row i.'
            )
            ds_c.attrs['sampling_rate_hz'] = TARGET_FS
            ds_c.attrs['segment_sec']      = SEG_SEC

            # /noisy
            ds_n = hf.create_dataset(
                'noisy', data=noisy_buf, dtype='float32',
                compression='gzip', compression_opts=4, chunks=chunk,
            )
            ds_n.attrs['description']     = (
                'Normalised contaminated EEG. '
                'shape=(N_segments, 512). '
                'Row i pairs with /clean row i.'
            )
            ds_n.attrs['sampling_rate_hz'] = TARGET_FS
            ds_n.attrs['segment_sec']      = SEG_SEC

            # file-level metadata
            hf.attrs['artifact_combo']  = combo
            hf.attrs['split']           = split_name
            hf.attrs['n_segments']      = total
            hf.attrs['n_eeg_segs']      = n_e
            hf.attrs['n_snr_levels']    = n_snr
            hf.attrs['snr_levels_db']   = str(snr_levels)
            hf.attrs['mixing_eq']       = 'xC = xP + sum_k(alpha_k * n_k)'
            hf.attrs['alpha_formula']   = 'alpha_k = RMS(xP)/(RMS(n_k)*10^(SNR_dB/10))'
            hf.attrs['normalisation']   = 'x_hat=x/std(noisy), y_hat=y/std(noisy)'
            hf.attrs['split_ratios']    = '80/10/10 train/val/test'
            hf.attrs['random_seed']     = RANDOM_SEED
            hf.attrs['ref_eegdnoisenet'] = 'Zhang et al. (2021) arXiv:2009.11662'
            hf.attrs['ref_mitbih']      = 'Moody & Mark (2001) IEEE Eng Med Biol Mag'
            hf.attrs['ref_ctdcenet']    = 'Tang et al. (2025) IEEE JBHI 10.1109/JBHI.2025.3535592'

        row_counts[split_name] = total
        size_mb = h5_path.stat().st_size / 1e6
        tqdm.write(
            f'    {split_name:<5s}: {total:>8,} rows '
            f'-> {h5_path.name}  ({size_mb:.1f} MB)'
        )

        del clean_buf, noisy_buf
        gc.collect()

    return row_counts


print('Builder function defined')

Builder function defined


## 6 — Generate all 7 x 3 = 21 HDF5 files

In [ ]:
ARTIFACT_ARRAYS = {'emg': emg, 'eog': eog, 'ecg': ecg}

rng     = np.random.default_rng(RANDOM_SEED)
summary = []

for combo in tqdm(ARTIFACT_COMBOS, desc='Building datasets'):
    label = '_'.join(combo)
    tqdm.write(f'\n  [{label}]')

    counts = build_and_save(
        eeg_arr    = eeg,
        art_names  = combo,
        art_arrays = ARTIFACT_ARRAYS,
        snr_levels = SNR_LEVELS,
        out_dir    = OUT_DIR,
        rng        = rng,
    )

    summary.append({
        'combo':       label,
        'n_artifacts': len(combo),
        'train_rows':  counts['train'],
        'val_rows':    counts['val'],
        'test_rows':   counts['test'],
        'total_rows':  sum(counts.values()),
    })

print('\nAll 21 HDF5 files generated')

Building datasets:   0%|          | 0/7 [00:00<?, ?it/s]


  [emg]
    train:   36,110 rows -> emg_train.h5  (137.6 MB)
    val  :    4,510 rows -> emg_val.h5  (17.2 MB)
    test :    4,520 rows -> emg_test.h5  (17.2 MB)

  [eog]
    train:   36,110 rows -> eog_train.h5  (137.7 MB)
    val  :    4,510 rows -> eog_val.h5  (17.2 MB)
    test :    4,520 rows -> eog_test.h5  (17.2 MB)

  [ecg]
    train:   36,110 rows -> ecg_train.h5  (136.5 MB)
    val  :    4,510 rows -> ecg_val.h5  (17.1 MB)
    test :    4,520 rows -> ecg_test.h5  (17.1 MB)

  [emg_eog]
    train:   36,110 rows -> emg_eog_train.h5  (137.6 MB)
    val  :    4,510 rows -> emg_eog_val.h5  (17.2 MB)
    test :    4,520 rows -> emg_eog_test.h5  (17.2 MB)

  [emg_ecg]
    train:   36,110 rows -> emg_ecg_train.h5  (137.1 MB)
    val  :    4,510 rows -> emg_ecg_val.h5  (17.1 MB)
    test :    4,520 rows -> emg_ecg_test.h5  (17.2 MB)

  [eog_ecg]
    train:   36,110 rows -> eog_ecg_train.h5  (137.0 MB)
    val  :    4,510 rows -> eog_ecg_val.h5  (17.1 MB)
    test :    4,520 rows -> e

## 7 — Summary table

In [ ]:
summary_df = pd.DataFrame(summary)

print('=== Dataset I -- HDF5 file summary ===')
print(summary_df.to_string(index=False))

h5_files   = sorted(OUT_DIR.glob('*.h5'))
total_mb   = sum(f.stat().st_size for f in h5_files) / 1e6

print(f'\nFiles generated : {len(h5_files)}  (7 combos x 3 splits = 21)')
print(f'Total disk size : {total_mb:.1f} MB  (gzip-4 compressed)')
print(f'Each file stores: /clean (N,512) float32  and  /noisy (N,512) float32')

summary_df.to_csv(OUT_DIR / 'dataset1_summary.csv', index=False)
print(f'Summary CSV     : {OUT_DIR}/dataset1_summary.csv')

## 8 — Verification

In [22]:
print('=== Read-back verification ===')
all_ok = True

for combo in ARTIFACT_COMBOS:
    label = '_'.join(combo)
    for split in ('train', 'val', 'test'):
        path = OUT_DIR / f'{label}_{split}.h5'

        if not path.exists():
            print(f'  MISSING: {path.name}')
            all_ok = False
            continue

        with h5py.File(path, 'r') as hf:
            c_shape = hf['clean'].shape
            n_shape = hf['noisy'].shape

            # shapes must match and have 512 columns
            ok_shape = (c_shape == n_shape) and (c_shape[1] == SEGMENT_LEN)
            if not ok_shape:
                print(f'  BAD SHAPE  {path.name}: clean={c_shape} noisy={n_shape}')
                all_ok = False
                continue

            # dtype
            ok_dtype = (hf['clean'].dtype == np.float32) and (hf['noisy'].dtype == np.float32)
            if not ok_dtype:
                print(f'  BAD DTYPE  {path.name}: clean={hf["clean"].dtype} noisy={hf["noisy"].dtype}')
                all_ok = False
                continue

            # normalisation: std(noisy row) should be ~1.0
            noisy_sample = hf['noisy'][:200]
            stds         = np.std(noisy_sample, axis=1)
            bad_norm     = np.abs(stds - 1.0) > 0.05
            if bad_norm.any():
                print(f'  NORM WARN  {path.name}: {bad_norm.sum()}/200 rows |std-1|>0.05')
                all_ok = False
                continue

            # metadata
            assert hf.attrs['artifact_combo'] == label, 'combo attribute mismatch'
            assert hf.attrs['split']          == split, 'split attribute mismatch'

        print(f'  OK  {path.name:<32s}  shape={c_shape}  dtype=float32')

if all_ok:
    print('\nAll 21 files verified: shapes, dtypes, normalisation, metadata -- all OK')
else:
    print('\nSome checks failed -- see messages above')

=== Read-back verification ===
  OK  emg_train.h5                      shape=(36110, 512)  dtype=float32
  OK  emg_val.h5                        shape=(4510, 512)  dtype=float32
  OK  emg_test.h5                       shape=(4520, 512)  dtype=float32
  OK  eog_train.h5                      shape=(36110, 512)  dtype=float32
  OK  eog_val.h5                        shape=(4510, 512)  dtype=float32
  OK  eog_test.h5                       shape=(4520, 512)  dtype=float32
  OK  ecg_train.h5                      shape=(36110, 512)  dtype=float32
  OK  ecg_val.h5                        shape=(4510, 512)  dtype=float32
  OK  ecg_test.h5                       shape=(4520, 512)  dtype=float32
  OK  emg_eog_train.h5                  shape=(36110, 512)  dtype=float32
  OK  emg_eog_val.h5                    shape=(4510, 512)  dtype=float32
  OK  emg_eog_test.h5                   shape=(4520, 512)  dtype=float32
  OK  emg_ecg_train.h5                  shape=(36110, 512)  dtype=float32
  OK  emg_ecg_v

## 9 — How to use these files in a training loop

In [23]:
# --- Quick read-back demo ---
demo_path = OUT_DIR / 'emg_train.h5'

with h5py.File(demo_path, 'r') as hf:
    print('File:', demo_path.name)
    print(f'  /clean  shape={hf["clean"].shape}  dtype={hf["clean"].dtype}')
    print(f'  /noisy  shape={hf["noisy"].shape}  dtype={hf["noisy"].dtype}')
    print('\nFile attributes:')
    for k, v in hf.attrs.items():
        print(f'  {k:<28s}: {v}')

    clean_ex = hf['clean'][:5]   # load 5 rows as a demo
    noisy_ex = hf['noisy'][:5]

print(f'\nRow 0 -- std(clean)={np.std(clean_ex[0]):.4f}  std(noisy)={np.std(noisy_ex[0]):.4f}')
print('         (std(noisy) should be 1.0 after normalisation)')

File: emg_train.h5
  /clean  shape=(36110, 512)  dtype=float32
  /noisy  shape=(36110, 512)  dtype=float32

File attributes:
  alpha_formula               : alpha_k = RMS(xP)/(RMS(n_k)*10^(SNR_dB/10))
  artifact_combo              : emg
  mixing_eq                   : xC = xP + sum_k(alpha_k * n_k)
  n_eeg_segs                  : 3611
  n_segments                  : 36110
  n_snr_levels                : 10
  normalisation               : x_hat=x/std(noisy), y_hat=y/std(noisy)
  random_seed                 : 42
  ref_ctdcenet                : Tang et al. (2025) IEEE JBHI 10.1109/JBHI.2025.3535592
  ref_eegdnoisenet            : Zhang et al. (2021) arXiv:2009.11662
  ref_mitbih                  : Moody & Mark (2001) IEEE Eng Med Biol Mag
  snr_levels_db               : [-7, -6, -5, -4, -3, -2, -1, 0, 1, 2]
  split                       : train
  split_ratios                : 80/10/10 train/val/test

Row 0 -- std(clean)=0.1953  std(noisy)=1.0000
         (std(noisy) should be 1.0 after no

In [ ]:
# --- PyTorch Dataset and DataLoader (copy-paste ready) ---
pytorch_code = '''
import h5py, torch
from torch.utils.data import Dataset, DataLoader


class EEGDenoiseH5Dataset(Dataset):
    """
    Lazy-loading PyTorch Dataset backed by an HDF5 file.

    Each HDF5 file contains:
      /clean  (N, 512) float32  -- normalised ground-truth EEG
      /noisy  (N, 512) float32  -- normalised contaminated EEG

    __getitem__ returns (noisy_input, clean_target) as (1,512) tensors
    (channel-first layout, ready for nn.Conv1d).
    """

    def __init__(self, h5_path: str):
        self._path = h5_path
        self._hf   = h5py.File(h5_path, 'r')
        self.clean = self._hf['clean']   # lazy proxy
        self.noisy = self._hf['noisy']   # lazy proxy
        self._len  = self.clean.shape[0]

    def __len__(self):
        return self._len

    def __getitem__(self, idx):
        c = torch.tensor(self.clean[idx], dtype=torch.float32).unsqueeze(0)  # (1,512)
        n = torch.tensor(self.noisy[idx], dtype=torch.float32).unsqueeze(0)  # (1,512)
        return n, c   # (noisy_input, clean_target)

    def close(self):
        self._hf.close()


# Usage -- swap 'emg' for any of the 7 combo labels
BASE = 'dataset1_h5'

train_ds = EEGDenoiseH5Dataset(f'{BASE}/emg_train.h5')
val_ds   = EEGDenoiseH5Dataset(f'{BASE}/emg_val.h5')
test_ds  = EEGDenoiseH5Dataset(f'{BASE}/emg_test.h5')

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True,  num_workers=4, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=32, shuffle=False, num_workers=4, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=32, shuffle=False, num_workers=4, pin_memory=True)

# Inspect one batch
noisy_batch, clean_batch = next(iter(train_loader))
print(noisy_batch.shape)   # torch.Size([32, 1, 512])
print(clean_batch.shape)   # torch.Size([32, 1, 512])
'''

print(pytorch_code)


import h5py, torch
from torch.utils.data import Dataset, DataLoader


class EEGDenoiseH5Dataset(Dataset):
    """
    Lazy-loading PyTorch Dataset backed by an HDF5 file.

    Each HDF5 file contains:
      /clean  (N, 512) float32  -- normalised ground-truth EEG
      /noisy  (N, 512) float32  -- normalised contaminated EEG

    __getitem__ returns (noisy_input, clean_target) as (1,512) tensors
    (channel-first layout, ready for nn.Conv1d).
    """

    def __init__(self, h5_path: str):
        self._path = h5_path
        self._hf   = h5py.File(h5_path, 'r')
        self.clean = self._hf['clean']   # lazy proxy
        self.noisy = self._hf['noisy']   # lazy proxy
        self._len  = self.clean.shape[0]

    def __len__(self):
        return self._len

    def __getitem__(self, idx):
        c = torch.tensor(self.clean[idx], dtype=torch.float32).unsqueeze(0)  # (1,512)
        n = torch.tensor(self.noisy[idx], dtype=torch.float32).unsqueeze(0)  # (1,512)
        return n, c   # (

## 10 — (Optional) Zip for download

In [ ]:
ZIP_PATH = Path('./dataset1_h5.zip')

with zipfile.ZipFile(ZIP_PATH, 'w', compression=zipfile.ZIP_STORED) as zf:
    # ZIP_STORED because HDF5 files are already gzip-compressed internally
    for f in sorted(OUT_DIR.iterdir()):
        zf.write(f, arcname=f.name)

zip_mb = ZIP_PATH.stat().st_size / 1e6
print(f'Archive: {ZIP_PATH.resolve()}  ({zip_mb:.1f} MB)')
print('  Colab : Files panel (left sidebar) -> right-click -> Download')
print('  Kaggle: Output tab -> dataset1_h5.zip')

Archive: /content/dataset1_h5.zip  (1201.4 MB)
  Colab : Files panel (left sidebar) -> right-click -> Download
  Kaggle: Output tab -> dataset1_h5.zip
